[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/02_00_main_Option1.ipynb)

In [ ]:
# --- Course setup (uncomment when running on Colab) ---
#!git clone https://github.com/tunnel-ai/way.git
#import sys; sys.path.insert(0, "/content/way/src")

# Module 2: Linear Regression (Option 1)

**Target:** `log1p(transaction_amount)`

This notebook is one of 2 walks through Module 2. `02_00_main_Option2.ipynb` predicts `transaction_loss_amount`, but that target is ~96% zeros with a heavy fraud tail, which makes linear regression structurally inappropriate (every learned model lands at negative R²). That is fun to see, but not the main point, so we'll run that only if time allows. Here we predict **transaction amount** instead, which is a textbook linear regression target:

- Continuous and well-distributed (after `log1p` transform)
- Real signal from features (merchant category, payment channel, account history)
- Coefficients that mean something
- Regularization that actually pays off

We walk through the standard linear regression workflow plus a tree-based comparison:

1. Frame the problem and check the target distribution
2. Define X/y, split, audit
3. Baselines
4. OLS with a real preprocessing pipeline
5. **Coefficient interpretation** (the part regression is *for*): including the merchant_id dominance and conditional-vs-marginal effects
6. Residual diagnostics
7. **Multicollinearity**: when regularization helps (and when it doesn't)
8. Ridge / Lasso / ElasticNet with cross-validation
9. **Beyond linear**: does a tree-based model do better, and how do feature importances compare to coefficients?
10. Takeaways

**Dataset:** the canonical synthetic transaction dataset, generated by `core.generators.transaction_risk_dgp.generate_transaction_risk_dataset(seed=1955)`.

In [ ]:
# --- Imports ------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 1955
np.random.seed(RANDOM_STATE)

In [ ]:
# --- Load canonical dataset ---------------------------------------------------
from core.generators.transaction_risk_dgp import generate_transaction_risk_dataset

df = generate_transaction_risk_dataset(seed=RANDOM_STATE)

print(df.shape)
df.head()

## 1) Why this target works for linear regression

Two practical questions tell you whether linear regression is the right tool:

1. **Is the target roughly symmetric / Gaussian (or transformable to it)?**
2. **Are there features with linear-ish relationships to the target?**

The raw `transaction_amount` is heavily right-skewed (a few large transactions dominate the right tail). The `log1p` transform compresses that tail and gives us a near-symmetric distribution... exactly what OLS assumes.

In [ ]:
TARGET = "transaction_amount"

amount = df[TARGET]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(amount, bins=100)
axes[0].set_title(f"Raw {TARGET}")
axes[0].set_xlabel("$")
axes[0].set_ylabel("count")

axes[1].hist(np.log1p(amount), bins=100)
axes[1].set_title(f"log1p({TARGET})")
axes[1].set_xlabel("log1p($)")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()

print(f"Raw skewness:    {amount.skew():.2f}")
print(f"log1p skewness:  {np.log1p(amount).skew():.2f}")

## 2) Define features, target, train/test split

We are predicting `log1p(transaction_amount)` from the rest of the row.

**What to drop:**
- `transaction_amount` itself (the target)
- `transaction_loss_amount`, proportional to amount when fraud occurs → leakage
- Label / post-event fields: `is_fraud`, `chargeback_flag`, `manual_review_score`, `fraud_probability_latent`
- Pure IDs: `transaction_id`, `account_id`
- `merchant_description`: 24K unique values, mostly text noise for this purpose
- `merchant_name`:  redundant with `merchant_id`

**What to keep, and how:**
- `merchant_id` cast to **string** so it routes through the categorical pipeline (high-cardinality, will use one-hot with `min_frequency=50`)
- All other categoricals: impute → one-hot encode
- Numerics: median impute → standardize (scaling matters once we add regularization)

In [ ]:
# --- Define features / target -------------------------------------------------
LEAKAGE_OR_LABEL = [
    "transaction_loss_amount",     # proportional to amount when fraud → leakage
    "is_fraud",                    # post-event label
    "chargeback_flag",             # post-event outcome
    "manual_review_score",         # post-event review
    "fraud_probability_latent",    # latent generator probability
]
ID_COLS = ["transaction_id", "account_id"]
DROP_TEXT = ["merchant_description", "merchant_name"]  # too high-cardinality / redundant

X = df.drop(columns=[TARGET] + LEAKAGE_OR_LABEL + ID_COLS + DROP_TEXT).copy()

# merchant_id is an integer in the dataset, but it's an arbitrary identifier, cast to
# string so the pipeline treats it as a category (not as a continuous integer scale).
X["merchant_id"] = X["merchant_id"].astype(str)

y = np.log1p(df[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

# Identify column types
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

print("Train:", X_train.shape, " Test:", X_test.shape)
print("\nNumeric features:")
for c in numeric_cols:
    print("  ", c)
print("\nCategorical features (n_unique):")
for c in categorical_cols:
    print(f"   {c:<25}  {X_train[c].nunique()}")

## 3) Baselines

Two log-space baselines:
- **Mean baseline**: always predict `mean(y_train)`
- **Median baseline**:al ways predict `median(y_train)` (more robust under skew)

Models that can't beat these aren't learning useful signal. Note that all metrics here are in **log space**: `MAE_log`, `RMSE_log`. To convert to dollar-space MAE, you'd apply `expm1` to predictions and recompute...we'll keep things in log space for now since that's where the linear-regression assumptions hold.

In [ ]:
def regression_report(y_true, y_pred, label="model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return pd.Series({"MAE_log": mae, "RMSE_log": rmse, "R2": r2}, name=label)

mean_pred = np.full_like(y_test, fill_value=float(y_train.mean()), dtype=float)
median_pred = np.full_like(y_test, fill_value=float(y_train.median()), dtype=float)

all_results = []
all_results.append(regression_report(y_test, mean_pred, "Baseline: mean"))
all_results.append(regression_report(y_test, median_pred, "Baseline: median"))

pd.concat(all_results, axis=1).T

## 4) OLS with preprocessing pipeline

Standard preprocessing pattern, all inside a single `Pipeline`. Transformations are fit on training data only:

- **Numeric** → median impute + standardize (`StandardScaler`)
- **Low-cardinality categoricals** (e.g. `payment_channel`, `country`, `merchant_category`) → impute + plain one-hot
- **High-cardinality `merchant_id`** → impute + one-hot with `min_frequency=50` (rare merchants get grouped into a single bucket so we don't blow up the feature count)

Wrapping all of this in a single `Pipeline` is really key here...it prevents data leakage (test-set statistics never touch fit) and makes cross-validation correct.

In [ ]:
# --- Preprocessing ------------------------------------------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

low_card_cols = [c for c in categorical_cols if c != "merchant_id"]
high_card_cols = ["merchant_id"]

low_card_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

high_card_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat_low", low_card_transformer, low_card_cols),
        ("cat_high", high_card_transformer, high_card_cols),
    ],
    remainder="drop",
)

ols_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LinearRegression()),
])

ols_model

In [ ]:
# --- Fit + evaluate -----------------------------------------------------------
ols_model.fit(X_train, y_train)
y_pred = ols_model.predict(X_test)

all_results.append(regression_report(y_test, y_pred, "OLS"))
pd.concat(all_results, axis=1).T

## 5) Coefficient interpretation

This is what linear regression is *for*. Because the target is `log1p(amount)`, every coefficient has a clean multiplicative interpretation:

$$\widehat{\text{amount}} \approx e^{\beta_0 + \sum_j \beta_j x_j} - 1$$

So if a coefficient $\beta_j$ is `0.30`, the presence (or +1 SD) of feature $j$ multiplies the expected amount by $e^{0.30} \approx 1.35$, a **35% increase**. A coefficient of `-0.40` means $e^{-0.40} \approx 0.67$, a **33% decrease**.

We pull out the largest positive and negative coefficients below.

In [ ]:
# --- Inspect coefficients -----------------------------------------------------
feature_names = ols_model.named_steps["preprocess"].get_feature_names_out()
coefs = ols_model.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs,
})
coef_df["pct_effect"] = (np.exp(coef_df["coef"]) - 1) * 100  # multiplicative effect on $

print("Top 10 features associated with HIGHER transaction amount:")
display(coef_df.nlargest(10, "coef").reset_index(drop=True).round(4))

print("\nTop 10 features associated with LOWER transaction amount:")
display(coef_df.nsmallest(10, "coef").reset_index(drop=True).round(4))

### What happened: high-cardinality categoricals dominate the coefficient table

Look at the top 10 above. Every row except `avg_transaction_amount_30d` is a `merchant_id` dummy,  anonymous IDs like `merchant_id_64` or `merchant_id_157` that carry **no semantic meaning**. We can read off "merchant 64 averages ~21% higher than baseline" but we can't say *why*... there's nothing to interpret about the ID itself.

Three mechanics are colliding:

1. **Cardinality asymmetry.** Even after `min_frequency=50` grouping, `merchant_id` produces *hundreds* of dummy columns. `merchant_category` has 18 levels, `country` has 10. The model has far more degrees of freedom in merchant_id, so it absorbs variance there.
2. **merchant_id contains within-category variation.** "Merchant 64" might be a premium brand inside `electronics`; "merchant 157" might be a budget brand. Once merchant_id is in the model, the `merchant_category` coefficient shrinks toward zero, merchant_id already encodes everything category does, plus more granularity.
3. **Ranking by magnitude favors the most numerous group.** With hundreds of merchant dummies and only ~50 interpretable features, the top 10 will probabilistically be merchant dummies.

The one interpretable result that survives the cut:

> **`avg_transaction_amount_30d`** at coefficient `+0.4279` → **+53.4%**. A 1-SD increase in an account's 30-day average amount predicts a ~53% higher current transaction. The dominant single signal is account-level spending history.

The rest of the interpretable signal is still in the model,  it just isn't in the top 10. Below we filter out the merchant_id dummies and look at what the model learned about **interpretable** features.

In [ ]:
# --- Coefficients excluding merchant_id ---------------------------------------
# Filter out the high-cardinality merchant_id dummies to reveal what the model
# learned about features whose coefficients we can actually narrate.

interpretable_mask = ~coef_df["feature"].str.startswith("cat_high__merchant_id_")
coef_interp = coef_df[interpretable_mask].copy()

print(f"Filtered out {(~interpretable_mask).sum()} merchant_id dummies; "
      f"{len(coef_interp)} interpretable features remain.\n")

print("Top 10 interpretable features (HIGHER amount):")
display(coef_interp.nlargest(10, "coef").reset_index(drop=True).round(4))

print("\nTop 10 interpretable features (LOWER amount):")
display(coef_interp.nsmallest(10, "coef").reset_index(drop=True).round(4))

### Two views, two purposes

You should now have a much more readable view: merchant categories, payment channels, country effects, device types, features whose coefficients you can actually narrate to a stakeholder.

**Practical takeaway:** in production you usually want both views.

- **Full coefficient table** (with merchant_id) tells you which **specific entities** the model treats as outliers. Useful for monitoring, fraud detection, ops alerting (e.g. "merchant 64 has unusually high transactions").
- **Interpretable subset** (without merchant_id) tells you what **behavioral patterns** the model has learned. Useful for stakeholder explanation, business intuition, and identifying what's structurally driving predictions (e.g. "luxury merchants are X% higher, education is Y% lower").

Neither view is "right", they answer different questions. This is also a recurring theme in modeling: **the choice of what to include in your feature set affects what your model can tell you**, not just how well it predicts. High-cardinality identifiers boost predictive performance but blur the explanatory story.

### Conditional vs. marginal: why the signs sometimes flip

The most counterintuitive entry above was `merchant_category_pharmacy` at **−2.26%**, surprising, because pharmacy transactions are actually **larger than average overall** (a raw `groupby` puts them at roughly +5.2% above the mean). How can the same category be *above* average in raw data but *below* average in the model? The answer is the difference between **marginal** and **conditional** interpretation:

- **Marginal effect** = "On average, how much higher are pharmacy transactions than the overall average?" Just a `groupby`...no model needed.
- **Conditional effect** (the regression coefficient) = "Holding everything else constant , including which specific merchant, how much higher are pharmacy transactions?"

The two answers can disagree dramatically when features are correlated. `merchant_id` already encodes which specific merchants are large pharmacy chains, so once it's in the model, the *remaining* signal in `merchant_category_pharmacy` is "what's left after controlling for merchant identity", and that residual is below baseline.

This is one of the most important and most-missed ideas in linear regression interpretation:

> **A regression coefficient is a partial effect, it answers what would happen if you changed *only* this feature, holding the rest fixed. Marginal averages from a `groupby` answer a different question entirely.**

Below we put both views side by side for every merchant category. Watch for **sign flips** (pharmacy, gas, education) and **magnitude collapses** (subscription, gaming, grocery, where merchant_id absorbs about half of what raw averages would attribute to the category).

In [ ]:
# --- Conditional vs marginal effects: side-by-side comparison ----------------

# Marginal: raw category mean vs the overall mean, in % terms. No model required.
overall_mean = df["transaction_amount"].mean()
marginal = (
    df.groupby("merchant_category")["transaction_amount"]
      .mean()
      .div(overall_mean)
      .sub(1)
      .mul(100)
      .rename("marginal_%_vs_overall")
)

# Conditional: from the OLS coefficient, expressed as a multiplicative % effect.
# These are the partial effects, what changes when only this feature changes.
conditional = (
    coef_df[coef_df["feature"].str.startswith("cat_low__merchant_category_")]
        .assign(category=lambda d: d["feature"]
                .str.replace("cat_low__merchant_category_", "", regex=False))
        .set_index("category")["pct_effect"]
        .rename("conditional_%_effect")
)

compare = (
    pd.concat([marginal, conditional], axis=1)
      .sort_values("marginal_%_vs_overall", ascending=False)
)
compare.round(2)

### Three patterns to recognize

The side-by-side comparison reveals three patterns whenever you compare conditional and marginal effects:

**1. Magnitude collapse (most common).** The conditional effect has the same sign as the marginal but is smaller, often by half. Examples here: `subscription` (+5.4 → +2.2), `gaming` (+4.9 → +2.7), `grocery` (+4.0 → +2.1), `financial_services` (−4.8 → −2.1). Interpretation: **another feature is sharing credit for the same variation.** In our case, `merchant_id` absorbs roughly half the explanatory work that the raw average attributes to `merchant_category`.

**2. Sign flip (rare but striking).** Marginal and conditional disagree about *direction*. Examples here: **pharmacy** (+5.2 → −2.3), **gas** (−5.0 → +1.9), **education** (−5.3 → +0.7), **travel** (+4.2 → −1.1). Interpretation: **the marginal is being driven by something other than the category itself.** For pharmacy, big-name chain merchants are doing all the work, once you control for those, the residual "pharmacy-ness" is below baseline. For gas, low-amount gas stations dominate the marginal, but the residual category effect is positive once specific stations are accounted for.

**3. Amplification (rarest).** The conditional effect is *larger* than the marginal, same sign, magnified. Here: `marketplace` (+1.3 → +4.8). Interpretation: **the category contains heterogeneous merchants** whose variation partially cancels out in the raw average, but the model can isolate the category effect once merchant identity is accounted for.

### Why this matters in practice

When marginal and conditional disagree, the right one to use depends on *which question you're answering*:

- **Reporting a finding to a non-technical stakeholder?** Use the **marginal**. "Pharmacy transactions average 5% higher than overall" is true, simple, and testable from raw data.
- **Predicting a single future transaction?** Use the **conditional**. The model's coefficient is what gets multiplied through during prediction.
- **Planning a policy intervention?** Use the **conditional** ...but be careful. "If we move a transaction from category A to B, expected amount changes by X" assumes you can hold every other feature (especially merchant_id) constant. If the intervention also changes which merchant is involved, the realized effect will be closer to the marginal.

Linear regression coefficients are partial effects, not raw differences. Conflating the two is one of the most common interpretation mistakes in applied modeling.

## 6) Residual diagnostics

Three plots tell you whether OLS assumptions are roughly satisfied:

1. **Predicted vs actual** points should hug the 45° line; systematic deviations reveal bias.
2. **Residuals vs predicted** should look like a structureless cloud centered on zero. Funnel shapes signal heteroskedasticity; curves signal nonlinearity.
3. **Residual histogram** should be roughly symmetric and centered at zero. Heavy tails or skew flag distributional violations.

In [ ]:
resid = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(y_pred, y_test, alpha=0.2, s=8)
lo, hi = float(y_test.min()), float(y_test.max())
axes[0].plot([lo, hi], [lo, hi], "r--", linewidth=1)
axes[0].set_title("Predicted vs actual (log space)")
axes[0].set_xlabel("Predicted log1p(amount)")
axes[0].set_ylabel("Actual log1p(amount)")

axes[1].scatter(y_pred, resid, alpha=0.2, s=8)
axes[1].axhline(0, color="r", linestyle="--", linewidth=1)
axes[1].set_title("Residuals vs predicted")
axes[1].set_xlabel("Predicted log1p(amount)")
axes[1].set_ylabel("Residual")

axes[2].hist(resid, bins=80)
axes[2].set_title("Residual distribution")
axes[2].set_xlabel("Residual")
axes[2].set_ylabel("count")

plt.tight_layout()
plt.show()

## 7) Multicollinearity: when regularization helps (and when it doesn't)

A common reason to reach for regularization is **multicollinearity**, that is when two or more features carry overlapping information. With correlated features, OLS coefficients can become unstable: small changes in the training data swing them in opposite directions while predictions barely move.  That's generally not good. Ridge (L2) penalizes large coefficients, forcing the model to *spread* effect across correlated features rather than putting all the weight on one of them.

But you have to **check** whether your features are actually correlated before assuming regularization will help. Below we verify directly, then cast a wider net across the numeric feature set.

In [ ]:
# --- Step 1: one pair we might expect to be correlated ---------------------------
hist_corr = X_train[["avg_transaction_amount_30d", "std_transaction_amount_30d"]].corr().iloc[0, 1]
print(f"corr(avg_30d, std_30d) = {hist_corr:+.3f}")

# --- Step 2: cast a wider net- go bigger-  what numeric pairs ARE correlated (|r| > 0.10)?
nm_corr = X_train[numeric_cols].corr()
pairs = (
    nm_corr.where(np.triu(np.ones_like(nm_corr, dtype=bool), k=1))   # upper triangle, no diagonal
           .stack()
           .pipe(lambda s: s[s.abs() > 0.10])
           .sort_values(key=abs, ascending=False)
           .round(3)
)

print("\nNumeric pairs with |corr| > 0.10:")
display(pairs.to_frame("correlation") if len(pairs) else "(none)")

# --- Step 3: do OLS and Ridge agree on coefficients across all numeric features?
coef_lookup = dict(zip(feature_names, coefs))
ridge_demo = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Ridge(alpha=10.0, random_state=RANDOM_STATE)),
])
ridge_demo.fit(X_train, y_train)
ridge_coefs = dict(zip(
    ridge_demo.named_steps["preprocess"].get_feature_names_out(),
    ridge_demo.named_steps["model"].coef_,
))

print("\nNumeric coefficients (scaled, log-space):")
print(f"  {'feature':<40} {'OLS':>10}   {'Ridge(α=10)':>12}")
for c in numeric_cols:
    feat = f"num__{c}"
    print(f"  {c:<40} {coef_lookup[feat]:+10.4f}   {ridge_coefs[feat]:+12.4f}")

### The read

Two things from the output:

1. **The pair we expected to be correlated isn't.** `corr(avg_30d, std_30d) ≈ 0`: they capture different aspects of transaction history (central tendency vs spread) and don't covary here. The intuition was wrong. (remember I made this dataset...so it says what I want to say, not what you expected, right?)
2. **There *are* correlated pairs, including a strong one** (`transactions_last_24h` ↔ `transactions_last_7d` at **0.856**). But OLS and Ridge give nearly identical coefficients across every numeric feature anyway. That's the empirical signature of "the correlation is real, but it isn't destabilizing OLS".. which means there's nothing for Ridge to fix! 

What is going on and why are our textbooks on statistics all so very very wrong? Why doesn't a 0.856 correlation cause OLS to misbehave the way the textbook story predicts? Because that story assumes a small or near-singular dataset. We have:

- **~95K training rows for ~12 numeric features** ...HUGE degrees of freedom
- **Standardized features**, a well-conditioned design matrix

With this much data and good scaling, OLS estimates stable coefficients even when features are strongly correlated. Ridge regularization mostly helps when one or both of those conditions break down (small sample, severe near-singularity).

**This doesn't mean Ridge is wasted.** Two reasons it can still earn its place in your pipeline:

- **No-cost insurance.** With a well-chosen `alpha`, Ridge does no harm even when OLS is already stable. Including it is cheap protection against future data drift or feature changes.
- **Lasso served a different role.** §8 will show Lasso zeroing out a chunk of one-hot dummies. That's not collinearity stabilization, it's **feature selection** on the long tail of merchant_id dummies that don't carry independent signal. Same regularization family, different mechanism, different problem solved.

**Practical takeaway:** match the tool to the actual problem.

- Regularization (Ridge) for genuine instability, usually small samples or near-singular features, not just moderate correlation.
- Sparse regularization (Lasso) for feature selection across many weak signals.
- A different model class (trees, §9) for nonlinear interactions.

Reaching for Ridge by default isn't wrong, it might not be necessary here, but its worth noting it doesn't solve problems the data doesn't have. Let that sink in. 

## 8) Regularization: Ridge / Lasso / Elastic Net

With many one-hot features (especially the merchant_id expansion) and some collinear numeric predictors, regularization gives more stable coefficients and often slightly better held-out performance.

- **Ridge (L2)**: shrinks all coefficients smoothly
- **Lasso (L1)**: drives some coefficients to *exactly zero* (built-in feature selection)
- **Elastic Net**: convex combination of L1 and L2

We tune each model's `alpha` (and ElasticNet's `l1_ratio`) via 5-fold cross-validation.

In [ ]:
# --- Cross-validated regularization -------------------------------------------
def make_reg_pipeline(regressor):
    return Pipeline(steps=[("preprocess", preprocess), ("model", regressor)])

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
alpha_grid = {"model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]}

ridge_search = GridSearchCV(
    make_reg_pipeline(Ridge(random_state=RANDOM_STATE)),
    param_grid=alpha_grid, cv=cv,
    scoring="neg_root_mean_squared_error", n_jobs=-1,
)
ridge_search.fit(X_train, y_train)

lasso_search = GridSearchCV(
    make_reg_pipeline(Lasso(max_iter=20000, random_state=RANDOM_STATE)),
    param_grid=alpha_grid, cv=cv,
    scoring="neg_root_mean_squared_error", n_jobs=-1,
)
lasso_search.fit(X_train, y_train)

enet_grid = {
    "model__alpha": [0.001, 0.01, 0.1, 1.0],
    "model__l1_ratio": [0.1, 0.5, 0.9],
}
enet_search = GridSearchCV(
    make_reg_pipeline(ElasticNet(max_iter=20000, random_state=RANDOM_STATE)),
    param_grid=enet_grid, cv=cv,
    scoring="neg_root_mean_squared_error", n_jobs=-1,
)
enet_search.fit(X_train, y_train)

print("Best Ridge:      ", ridge_search.best_params_)
print("Best Lasso:      ", lasso_search.best_params_)
print("Best ElasticNet: ", enet_search.best_params_)

In [ ]:
# --- Evaluate tuned models on test set ----------------------------------------
all_results.append(regression_report(y_test, ridge_search.predict(X_test), "Ridge (CV)"))
all_results.append(regression_report(y_test, lasso_search.predict(X_test), "Lasso (CV)"))
all_results.append(regression_report(y_test, enet_search.predict(X_test), "ElasticNet (CV)"))

# How many features did Lasso zero out?
lasso_coefs = lasso_search.best_estimator_.named_steps["model"].coef_
n_kept = int((lasso_coefs != 0).sum())
n_total = len(lasso_coefs)
print(f"Lasso kept {n_kept} of {n_total} features ({n_total - n_kept} zeroed out)\n")

pd.concat(all_results, axis=1).T

## 9) Beyond linear:  Does a nonlinear model do better? (tldr: not really)

Linear regression assumes the target is an **additive linear combination** of features. That works when relationships are smooth and effects don't depend on each other, but it can't capture **interactions** (e.g., "the effect of `merchant_category` depends on `country`") or **nonlinearities** (e.g., "transaction amount grows with account history but flattens above some threshold").

**Tree-based methods** make none of those assumptions. A decision tree splits the feature space recursively, automatically discovering interactions and nonlinearities. Random forests average many trees together to reduce variance and usually push performance further.

Two questions this section answers:

1. **Do tree methods do meaningfully better here?** That tells us whether linear was leaving signal on the table.
2. **What do they say about feature importance?** That gives us a second interpretation lens to compare against the OLS coefficients we worked through in §5.

We'll keep this brief, Module 3 covers tree methods more thoroughly in the classification context. Here it's a sanity check: does linear do enough for *this* regression problem?

In [ ]:
# --- Single decision tree -----------------------------------------------------
# Trees don't need scaling or one-hot encoding strictly, they split on raw
# feature values. We reuse the same `preprocess` for an apples-to-apples
# comparison; in practice you'd often skip scaling for trees.
tree_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", DecisionTreeRegressor(
        max_depth=8,                 # cap depth to prevent overfitting on training data
        min_samples_leaf=50,         # prevent splits on tiny subgroups
        random_state=RANDOM_STATE,
    )),
])

tree_model.fit(X_train, y_train)
all_results.append(regression_report(y_test, tree_model.predict(X_test), "Decision Tree (depth=8)"))
pd.concat(all_results, axis=1).T

In [ ]:
# --- Random forest ------------------------------------------------------------
# An ensemble of decision trees, each fit on a bootstrap sample with random
# feature subsets at each split. Averaging across trees reduces variance and
# usually beats a single tree by a meaningful margin.
#
# Note: this cell takes 30-60s to fit on ~95K rows × hundreds of one-hot dummies.
rf_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=10,                # modest depth keeps training fast
        min_samples_leaf=20,
        n_jobs=-1,                   # use all cores
        random_state=RANDOM_STATE,
    )),
])

rf_model.fit(X_train, y_train)
all_results.append(regression_report(y_test, rf_model.predict(X_test), "Random Forest"))
pd.concat(all_results, axis=1).T

In [ ]:
# --- Feature importance: what does the forest think matters? -----------------
# RandomForestRegressor exposes `feature_importances_`, a measure of how
# often each feature was used to split, weighted by the variance reduction
# it produced. Big number = "the model leans heavily on this feature."

rf_feature_names = rf_model.named_steps["preprocess"].get_feature_names_out()
rf_importances = rf_model.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": rf_importances,
})

print("Top 10 features (raw- same merchant_id dominance pattern as OLS?):")
display(importance_df.nlargest(10, "importance").reset_index(drop=True).round(4))

# The interpretable view, mirroring what we did for OLS coefficients in §5
interp_mask = ~importance_df["feature"].str.startswith("cat_high__merchant_id_")
print("\nTop 10 features (excluding merchant_id):")
display(importance_df[interp_mask].nlargest(10, "importance").reset_index(drop=True).round(4))

### What this table is telling you (and how it disagrees with §5)

Look at the top of the importance table: **`avg_transaction_amount_30d` accounts for ~89% of the model's predictive work.** The next eight numeric features pick up almost all of the rest, each contributing 1-2% individually. The categoricals: `merchant_category`, `payment_channel`, `country`, `device_type`, collectively contribute well under 1%.

Now compare to the OLS coefficient table from §5. There, **9 of the top 10 entries were `merchant_id` dummies**. Here, **zero merchant_ids** appear in the top 10. The two methods *completely disagree* on whether merchant identity matters. What a mess...

Why the divergence?

| | OLS | Random Forest |
|---|---|---|
| How effects work | **Additive**:each one-hot gets its own coefficient | **Hierarchical** : features compete to split on |
| Treats `avg_transaction_amount_30d` as | One of many features | **Primary splitter**: captures most account-level variation |
| Treats `merchant_id` as | Hundreds of additive coefficients | Rarely used because `avg_30d` already encodes per-account behavior |

This is the **correlated-features-splitting-credit** story from §7, played out in extreme form. `avg_transaction_amount_30d` is essentially a *learned per-account intercept*, accounts with high historical averages shop at specific merchants. OLS, being linear, has to distribute credit across hundreds of merchant dummies. RF, with hierarchical splits, splits on the per-account average and inherits the merchant-identity effect for free.

The takeaway is bigger than this one model:

> **When two features are correlated, which one "wins" credit depends on the model class.** Linear models distribute credit additively across all correlated features. Tree models give it to whichever feature splits cleanest first. There is no single "correct" answer to "which feature matters most", only a model-conditional answer.

### What this implies about the problem itself

The RF story is essentially: "If you only kept `avg_transaction_amount_30d`, you'd already explain ~89% of what the full feature set does." That's a meaningful structural finding:

> **This is mostly an account-level problem, not a transaction-level problem.** Established accounts are highly predictable from their own history. New accounts with no history are essentially unpredictable.

That raises three follow-up questions worth carrying into Module 3 and beyond:

- **Cold-start**: how do you predict for new accounts where `avg_30d` is missing or unreliable?
- **Two-stage modeling**: predict an account-level spending profile first, then model per-transaction residual variation. (This decomposition handles cold-start and gives you a cleaner per-transaction signal.)
- **What's the per-transaction signal?** The remaining ~11% of importance is what makes a given transaction differ from its account's average, captured by `time_since_last_transaction`, `account_age_days`, `merchant_risk_score`, and the velocity features. That's the *residual* signal worth modeling once account-level baseline is accounted for.

This kind of structural insight is **only** visible when you compare interpretation tools across model classes. Coefficients alone or importances alone wouldn't have surfaced it.

### Coefficients vs feature importance: two different questions

You now have two interpretation tools, and they answer fundamentally different questions:

| Tool | Question it answers | Direction info? |
|---|---|---|
| **OLS coefficient** (β) | "If I change this feature by 1 unit holding others fixed, what changes?", a *partial effect* with a sign. Multiplicative under our `log1p` target. | Yes, sign tells you up or down |
| **RF feature importance** | "How much does this feature contribute to the model's overall predictions?", a *total contribution*. | No, direction requires partial-dependence plots or SHAP |

Coefficients tell you *direction and magnitude under counterfactuals*. Feature importances tell you *which features the model depends on most*. They often agree on which features matter, but they say different things about *how* they matter.

For example: `transactions_last_7d` showed up in both views, but in §5 you saw it had a **negative** coefficient (more transactions → smaller per-transaction amount). Feature importance alone can't tell you that...it only tells you the model uses that feature heavily.

### When to reach for trees over linear regression

The comparison above gives you the answer for *this* problem. The general rule:

- **If RF beats linear by a small margin (~0.01–0.02 R²)**: linear was nearly enough. The interpretability and stability of OLS/Ridge are usually worth the small performance loss.
- **If RF beats linear by a large margin (~0.05+ R²)**: there are real interactions or nonlinearities the linear model can't capture. Either engineer those features in (interactions, polynomial terms, target-encoded categoricals) or ship the tree-based model and explain it with partial-dependence plots / SHAP.
- **If the gap is in between**: it depends on what you need to *do* with the model. Production-stable predictions that need to be explained to non-technical stakeholders → linear. Highest possible accuracy where each prediction can be examined individually → tree.

Module 3 covers tree-based methods in more depth (with classification). Here we just wanted to know: **does linear leave signal on the table for this regression problem?** Look at the gap between the best linear model and Random Forest above to read off the answer.

## 10) Takeaways

- **Choice of target matters more than choice of model.** With `log1p(transaction_amount)`, linear regression is in its element: continuous target, real signal, interpretable coefficients. The same dataset with a different target (`transaction_loss_amount`) lands every linear model in negative-R² territory.
- **Always check the target distribution first.** Heavy right-skew is a red flag for raw OLS. `log1p` is a good first move for monetary or count-like targets, and the residual diagnostics will tell you if it worked.
- **Coefficients tell a story.** With one-hot encoded categories and a log target, you can read off relative effects: "marketplace transactions are X% higher on average," "card-present transactions are Y% lower than online." This interpretability is *the* reason linear regression survives in production despite fancier alternatives.
- **Marginal ≠ conditional.** Regression coefficients are *partial effects*, what changes if you change *only* this feature, holding the rest fixed. That is useful, right? Raw `groupby` averages answer a different question (and frequently disagree). Be explicit about which one you're reporting.
- **Correlation isn't always instability.** Textbooks say correlated features destabilize OLS coefficients, but with large samples and standardized features (as here), OLS estimates stay stable even at `|corr| > 0.85`. Reach for Ridge when you genuinely have small-sample or near-singular situations, not just because two features happen to correlate.
- **Lasso doubles as feature selection.** When `alpha > 0`, Lasso drove a chunk of one-hot dummies to exactly zero,  telling you which merchant categories or countries don't carry independent signal once others are accounted for. That's a different mechanism (sparsity) than Ridge's stabilization, solving a different problem.
- **Linear vs nonlinear is a deliberate choice.** Random Forest gives you a sanity check on whether linear is leaving signal on the table. Use the gap to decide whether to engineer features and stay linear, or ship a tree model and accept the interpretive complexity.
- **Different model classes give different "what matters" stories.** OLS coefficients distributed credit across hundreds of merchant_id dummies. RF feature importance gave 90% to a single feature (`avg_transaction_amount_30d`). Neither is wrong , they answer different questions about the same data.

### Check in

1. What's OLS's test R², and how much (if at all) did regularization improve on it?
2. Pick one feature with a large positive coefficient. Does the direction make sense given what you know about transaction behavior?
3. Did Lasso zero out any features you would have expected to keep? Any features it *kept* that surprise you?
4. Did Random Forest meaningfully beat the best linear model? If yes, by how much? What would that tell you about the structure of the signal?
5. Compare the top RF feature-importance list to the top OLS interpretable-coefficient list. Are they roughly the same set? If features appear in one but not the other, what might that mean?
6. If you had to ship one model, which would it be. Why?